# OpenMetadata AI Agent
### Built by Baibhav Prateek | OpenMetadata Hackathon 2026

## What is this?
This notebook builds an intelligent AI agent that automatically 
decides how to search OpenMetadata based on your question.

## How it works:
1) You ask a question in plain English
2) The AI agent decides which tool to use
3) It fetches the right data from OpenMetadata
4) It gives you a clear and  helpful answers

## Why I built this:
Most data catalog tools require you to know exactly what to search for
This agent makes it conversational ,just ask naturally!!!

In [ ]:

import requests
import json
from groq import Groq

# Your credentials
GROQ_API_KEY = "paste api key here"
BASE_URL = "https://sandbox.open-metadata.org"
TOKEN = "paste your tokens here"

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

client = Groq(api_key=GROQ_API_KEY)
print("✅ Agent setup complete!")

In [ ]:
# These are the tools my agent can use

# Tool 1: Get a list of tables

def tool_get_tables(limit=10):
    """Get list of tables from OpenMetadata"""
    try:
        response = requests.get(
            f"{BASE_URL}/api/v1/tables",
            headers=HEADERS,
            params={"limit": limit}
        )
        if response.status_code != 200:
            return []
        tables = response.json().get("data", [])
        return [{"name": t.get("name"),
                 "description": t.get("description", "No description"),
                 "columns": len(t.get("columns", []))}
                for t in tables]
    except Exception as e:
        print(f"❌ Error fetching tables: {e}")
        return []
    
# Tool 2: Search for specific tables by keyword  


def tool_search_tables(keyword):
    """Search for specific tables by keyword"""
    try:
        response = requests.get(
            f"{BASE_URL}/api/v1/search/query",
            headers=HEADERS,
            params={"q": keyword, "index": "table_search_index", "limit": 5}
        )
        if response.status_code != 200:
            return []
        hits = response.json().get("hits", {}).get("hits", [])
        return [h.get("_source", {}).get("name", "") for h in hits]
    except Exception as e:
        print(f"❌ Error searching tables: {e}")
        return []


# Tool 3: Get all databases

def tool_get_databases():
    """Get all databases"""
    try:
        response = requests.get(
            f"{BASE_URL}/api/v1/databases",
            headers=HEADERS,
            params={"limit": 10}
        )
        if response.status_code != 200:
            return []
        dbs = response.json().get("data", [])
        return [d.get("name") for d in dbs]
    except Exception as e:
        print(f"❌ Error fetching databases: {e}")
        return []


print("✅ Agent tools ready!")

In [ ]:
# This is the brain of the agent
# First it thinks about which tool to use
# Then it uses that tool to fetch data
# Finally it gives a human-friendly answer
def run_agent(user_question):
    print(f"🧠 Agent thinking about: {user_question}")
    print("-" * 50)
    
    # Step 1: Ask AI to decide what to do
    decision_prompt = f"""You are a data catalog agent. 
You have these tools available:
1. get_tables - fetches list of tables
2. search_tables - searches tables by keyword
3. get_databases - fetches all databases

User question: {user_question}

Which tool should you use first? Reply with ONLY one of:
get_tables
search_tables: <keyword>
get_databases"""

    decision = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": decision_prompt}]
    ).choices[0].message.content.strip()
    
    print(f"🔧 Agent decided to use: {decision}")
    
    # Step 2: Execute the chosen tool
    if "search_tables" in decision:
        keyword = decision.split(":")[-1].strip()
        data = tool_search_tables(keyword)
        tool_used = f"search_tables('{keyword}')"
    elif "get_databases" in decision:
        data = tool_get_databases()
        tool_used = "get_databases()"
    else:
        data = tool_get_tables()
        tool_used = "get_tables()"
    
    print(f"📦 Data fetched: {data}")
    print("-" * 50)
    
    # Step 3: Ask AI to answer using the fetched data
    final_prompt = f"""You are a helpful data catalog assistant.
User asked: {user_question}
You used tool: {tool_used}
Data retrieved: {data}

Now answer the user's question clearly and helpfully."""

    final_answer = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}]
    ).choices[0].message.content

    print(f"🤖 Agent answer: {final_answer}")
    return final_answer

print("✅ Agent brain ready!")

In [ ]:
# Test the agent with different questions
test_questions = [
    "Which databases contain customer information?",
    "Find all tables that might have sensitive financial data",
    "What tables should a new data analyst explore first?"
]

print("=" * 60)
print("   🤖 My OpenMetadata AI Agent")
print("   Built for OpenMetadata Hackathon 2026")
print("=" * 60)

for question in test_questions:
    print(f"\n❓ User: {question}")
    print("=" * 60)
    run_agent(question)
    print()

print("✅ Agent demo complete!")